<a href="https://colab.research.google.com/github/Gopalchalak/Deep-Learning/blob/main/HyperParamterTunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('sample_data/diabetes.csv')

In [8]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
df.isnull().sum()
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [10]:
x = df.iloc[:,:-1].values
y = df.iloc[:, -1].values

In [11]:
from sklearn.preprocessing import StandardScaler


In [12]:
scaler = StandardScaler()


In [13]:
x = scaler.fit_transform(x)

In [14]:
x.shape

(768, 8)

In [15]:
from sklearn.model_selection import train_test_split
x_train , x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=1)

In [16]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [17]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.fit(x_train, y_train, batch_size=32, epochs=100, validation_data=(x_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5000 - loss: 0.7088 - val_accuracy: 0.6104 - val_loss: 0.6623
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6319 - loss: 0.6480 - val_accuracy: 0.7208 - val_loss: 0.6113
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7182 - loss: 0.6018 - val_accuracy: 0.7597 - val_loss: 0.5743
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7459 - loss: 0.5693 - val_accuracy: 0.7922 - val_loss: 0.5468
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7541 - loss: 0.5434 - val_accuracy: 0.7922 - val_loss: 0.5269
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7638 - loss: 0.5248 - val_accuracy: 0.7922 - val_loss: 0.5122
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.5105 - val_accuracy: 0.7922 - val_loss: 0.5017
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4996 - val_accuracy: 0.7857 - 

In [19]:
#how to select approptiate aoptimizer
# no. of nodes in a layer
# how to select no. of layers
# all in all one model

In [20]:
pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.0 MB/s eta 0:00:00


In [21]:
#how to select approptiate optimizer
import kerastuner as kt

/tmp/ipykernel_184/3321109293.py:2: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [22]:
def build_model1(hp):
  model = Sequential()
  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))
  optimizer = hp.Choice('optimizer', values = ['adam', 'sgd', 'rmsprop', 'adadelta'])
  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [23]:
# from keras_tuner.src.engine import objective
tuner = kt.RandomSearch(build_model1, objective= 'val_accuracy', max_trials=5)

In [24]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 19s


In [25]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [26]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
model.fit(x_train, y_train, batch_size=32, epochs=100, initial_epoch= 6, validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7590 - loss: 0.5114 - val_accuracy: 0.7922 - val_loss: 0.4949
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.4987 - val_accuracy: 0.7922 - val_loss: 0.4879
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.4907 - val_accuracy: 0.7987 - val_loss: 0.4846
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4851 - val_accuracy: 0.7987 - val_loss: 0.4811
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7704 - loss: 0.4794 - val_accuracy: 0.8052 - val_loss: 0.4791
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7785 - loss: 0.4748 - val_accuracy: 0.8052 - val_loss: 0.4781
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7850 - loss: 0.4716 - val_accuracy: 0.8052 - val_loss: 0.4764
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7736 - loss: 0.4686 - val_accuracy: 

In [29]:
# no. of nodes in a layer

def build_model2(hp):
  model = Sequential()
  units = hp.Int('units', min_value = 8, max_value = 128,step=8)
  model.add(Dense(units=units, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer="Adam", loss='binary_crossentropy', metrics=['accuracy'])


  return model

In [30]:
tuner = kt.RandomSearch(build_model2, objective= 'val_accuracy', max_trials=5, directory = 'mydir')

In [ ]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model1 = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(x_test, y_test))

In [ ]:
def build_model3(hp):
  model3 = Sequential()
  model3.add(Dense(32, activation='relu', input_dim=8))
  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    model3.add(Dense(32, activation='relu'))

  model3.add(Dense(1, activation='sigmoid'))

  model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model3

In [ ]:
tuner = kt.RandomSearch(build_model3, objective='val_accuracy', max_trials=3, directory='mydir', project_name='myProject')

In [ ]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(x_train, y_train, epochs=100, initial_epoch=6, validation_data=(x_test, y_test))

In [ ]:
#how to select approptiate aoptimizer
# no. of nodes in a layer
# how to select no. of layers
# all in all one model

def build_model4(hp):
  model4 = Sequential()

  units = hp.Int('units', min_value = 8, max_value = 128,step=8)
  model4.add(Dense(units=units, activation='relu', input_dim=8))
  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    model4.add(Dense(32, activation='relu'))
  model4.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values = ['adam', 'sgd', 'rmsprop', 'adadelta'])
  model4.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model4

In [ ]:
def build_model5(hp):
  model5 = Sequential()

  counter = 0

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    if counter ==0:
      model5.add(Dense(hp.Int(' units' + str(i), min_value=8, max_value=128, step=8), activation=hp.Choice('activation', values=['relu', 'sigmoid', 'tanh']), input_dim=8))
      model5.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9 ])))
    else:
      model5.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8), activation=hp.Choice('activation', values=['relu', 'sigmoid', 'tanh'])))
      model5.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9 ])))
    counter+=1

  model5.add(Dense(1, activation='sigmoid'))

  model5.compile(optimizer=hp.Choice('optimizer', values = ['adam', 'rmsprop', 'sgd', 'nadam', 'adadelta']),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])
  return model5

In [ ]:
tuner = kt.RandomSearch(build_model5, objective='val_accuracy', max_trials=3, directory='my_project_dir1', project_name="final1")

In [ ]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(x_train, y_train, epochs=200, initial_epoch=6, validation_data=(x_test, y_test))